In [1]:
import pandas as pd
import sqlite3
from datetime import datetime

In [2]:
df = pd.read_csv('../data/online_retail_cleaned.csv', parse_dates=['InvoiceDate', 'Date'])

In [3]:
print("Loaded cleaned data:", df.shape)
print(df.dtypes)

Loaded cleaned data: (805549, 14)
Invoice                 int64
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID              int64
Country                object
TotalPrice            float64
IsCancelled              bool
MonthYear              object
Date           datetime64[ns]
DayOfWeek              object
Hour                    int64
dtype: object


In [4]:
df['Date'] = pd.to_datetime(df['Date'])

In [5]:
dim_date = df[['Date']].drop_duplicates().sort_values('Date').reset_index(drop=True)

dim_date['Year']       = dim_date['Date'].dt.year
dim_date['Quarter']    = dim_date['Date'].dt.quarter
dim_date['Month']      = dim_date['Date'].dt.month
dim_date['MonthName']  = dim_date['Date'].dt.month_name()
dim_date['MonthYear']  = dim_date['Date'].dt.to_period('M').astype(str)
dim_date['Week']       = dim_date['Date'].dt.isocalendar().week
dim_date['DayOfWeek']  = dim_date['Date'].dt.day_name()
dim_date['DayOfMonth'] = dim_date['Date'].dt.day
dim_date['IsWeekend']  = dim_date['DayOfWeek'].isin(['Saturday', 'Sunday']).astype(int)

In [8]:
dim_date['DateKey'] = dim_date['Date'].dt.strftime('%Y%m%d').astype(int)

print("dim_date preview:")
display(dim_date.head())

dim_date preview:


,Date,Year,Quarter,Month,MonthName,MonthYear,Week,DayOfWeek,DayOfMonth,IsWeekend,DateKey
0,2009-12-01,2009,4,12,December,2009-12,49,Tuesday,1,0,20091201
1,2009-12-02,2009,4,12,December,2009-12,49,Wednesday,2,0,20091202
2,2009-12-03,2009,4,12,December,2009-12,49,Thursday,3,0,20091203
3,2009-12-04,2009,4,12,December,2009-12,49,Friday,4,0,20091204
4,2009-12-05,2009,4,12,December,2009-12,49,Saturday,5,1,20091205


In [9]:
dim_product = df[['StockCode', 'Description']].drop_duplicates()

In [10]:
dim_product['Description'] = dim_product['Description'].str.strip().str.title()

In [11]:
dim_product = dim_product.reset_index(drop=True)
dim_product['ProductKey'] = dim_product.index + 1

print("dim_product preview (top 10):")
display(dim_product.head(10))

dim_product preview (top 10):


,StockCode,Description,ProductKey
0,85048,15Cm Christmas Glass Ball 20 Lights,1
1,79323P,Pink Cherry Lights,2
2,79323W,White Cherry Lights,3
3,22041,"Record Frame 7"" Single Size",4
4,21232,Strawberry Ceramic Trinket Box,5
5,22064,Pink Doughnut Trinket Pot,6
6,21871,Save The Planet Mug,7
7,21523,Fancy Font Home Sweet Home Doormat,8
8,22350,Cat Bowl,9
9,22349,"Dog Bowl , Chasing Ball Design",10


In [12]:
dim_customer = df[['CustomerID', 'Country']].drop_duplicates()

dim_customer = dim_customer.reset_index(drop=True)
dim_customer['CustomerKey'] = dim_customer.index + 1

print("dim_customer preview:")
display(dim_customer.head())

dim_customer preview:


,CustomerID,Country,CustomerKey
0,13085,United Kingdom,1
1,13078,United Kingdom,2
2,15362,United Kingdom,3
3,18102,United Kingdom,4
4,12682,France,5


In [13]:
fact_sales = df.copy()

In [14]:
fact_sales = fact_sales.merge(dim_date[['Date', 'DateKey']], on='Date', how='left')
fact_sales = fact_sales.merge(dim_product[['StockCode', 'ProductKey']], on='StockCode', how='left')
fact_sales = fact_sales.merge(dim_customer[['CustomerID', 'CustomerKey']], on='CustomerID', how='left')

In [15]:
fact_sales = fact_sales[[
    'DateKey', 'CustomerKey', 'ProductKey',
    'Invoice', 'Quantity', 'UnitPrice', 'TotalPrice',
    'Hour', 'DayOfWeek'  
]]

print("fact_sales preview:")
display(fact_sales.head())

fact_sales preview:


,DateKey,CustomerKey,ProductKey,Invoice,Quantity,UnitPrice,TotalPrice,Hour,DayOfWeek
0,20091201,1,1,489434,12,6.95,83.4,7,Tuesday
1,20091201,1,2,489434,12,6.75,81.0,7,Tuesday
2,20091201,1,3,489434,12,6.75,81.0,7,Tuesday
3,20091201,1,3218,489434,12,6.75,81.0,7,Tuesday
4,20091201,1,4,489434,48,2.10,100.8,7,Tuesday


In [16]:
DB_PATH = '../data/ecommerce_analytics.db'

In [17]:
conn = sqlite3.connect(DB_PATH)

In [18]:
dim_date.to_sql('dim_date', conn, if_exists='replace', index=False)
dim_product.to_sql('dim_product', conn, if_exists='replace', index=False)
dim_customer.to_sql('dim_customer', conn, if_exists='replace', index=False)
fact_sales.to_sql('fact_sales', conn, if_exists='replace', index=False)

conn.close()

print(f"SQLite database created → {DB_PATH}")
print("Tables: dim_date, dim_product, dim_customer, fact_sales")

SQLite database created → ../data/ecommerce_analytics.db
Tables: dim_date, dim_product, dim_customer, fact_sales


In [ ]:
4